# BƯỚC 5: Tích hợp Pipeline & Inference Cuối cùng
## Integration Pipeline - End-to-End Hybrid AI System

Gộp tất cả module (YOLO + Feature Extraction + ML) thành pipeline hoàn chỉnh
- YOLO detect cell → Crop → Extract features → ML predict
- Visualize result trên ảnh gốc

## 1. Setup & Load All Models

## 2. Load All Pre-trained Models

## 3. Feature Extraction Function (từ Bước 3)

In [ ]:
class CellFeatureExtractor:
    """Trích xuất đặc trưng từ ảnh cell"""
    
    @staticmethod
    def extract_morphology_features(binary_img):
        features = {}
        area = cv2.countNonZero(binary_img)
        features['area'] = area
        
        contours, _ = cv2.findContours(binary_img, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        if len(contours) > 0:
            cnt = max(contours, key=cv2.contourArea)
            perimeter = cv2.arcLength(cnt, True)
            features['perimeter'] = perimeter
            
            if perimeter > 0:
                circularity = 4 * np.pi * area / (perimeter ** 2)
                features['circularity'] = circularity
            
            if len(cnt) >= 5:
                ellipse = cv2.fitEllipse(cnt)
                (x, y), (MA, ma), angle = ellipse
                if MA > 0:
                    eccentricity = np.sqrt(1 - (ma / MA) ** 2)
                    features['eccentricity'] = eccentricity
        
        return features
    
    @staticmethod
    def extract_color_features(img):
        features = {}
        features['mean_B'] = np.mean(img[:,:,0])
        features['mean_G'] = np.mean(img[:,:,1])
        features['mean_R'] = np.mean(img[:,:,2])
        features['std_B'] = np.std(img[:,:,0])
        features['std_G'] = np.std(img[:,:,1])
        features['std_R'] = np.std(img[:,:,2])
        
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(float)
        features['mean_H'] = np.mean(hsv[:,:,0])
        features['mean_S'] = np.mean(hsv[:,:,1])
        features['mean_V'] = np.mean(hsv[:,:,2])
        
        return features
    
    @staticmethod
    def extract_texture_features(gray_img):
        features = {}
        edges = cv2.Canny(gray_img, 50, 150)
        features['edge_density'] = np.sum(edges > 0) / edges.size
        
        hist = cv2.calcHist([gray_img], [0], None, [256], [0, 256])
        hist = hist.flatten() / hist.sum()
        entropy = -np.sum(hist[hist > 0] * np.log2(hist[hist > 0]))
        features['entropy'] = entropy
        features['contrast'] = np.std(gray_img)
        
        return features
    
    @staticmethod
    def extract(img):
        features = {}
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
        
        features.update(CellFeatureExtractor.extract_morphology_features(binary))
        features.update(CellFeatureExtractor.extract_color_features(img))
        features.update(CellFeatureExtractor.extract_texture_features(gray))
        
        return features

print("✓ Feature Extractor class defined")

## 4. Complete Pipeline Function

In [ ]:
class HybridAIPipeline:
    """End-to-end Hybrid AI Pipeline"""
    
    def __init__(self, yolo_model, ml_model, scaler):
        self.yolo_model = yolo_model
        self.ml_model = ml_model
        self.scaler = scaler
        self.feature_extractor = CellFeatureExtractor()
    
    def predict(self, image_path, conf_threshold=0.25):
        """Inference trên ảnh"""
        
        # Load ảnh
        img = cv2.imread(str(image_path))
        if img is None:
            return None, []
        
        h, w = img.shape[:2]
        result_img = img.copy()
        detections = []
        
        # Step 1: YOLO Detection
        results = self.yolo_model.predict(str(image_path), conf=conf_threshold, verbose=False)
        
        # Step 2: For each detection
        for box in results[0].boxes:
            # Get coordinates
            x1, y1, x2, y2 = box.xyxy[0].int().tolist()
            conf = box.conf[0].item()
            
            # Crop
            crop = img[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            
            # Step 3: Extract features
            features = self.feature_extractor.extract(crop)
            features_array = np.array([list(features.values())])
            features_scaled = self.scaler.transform(features_array)
            
            # Step 4: ML Prediction
            prediction = self.ml_model.predict(features_scaled)[0]
            if hasattr(self.ml_model, 'predict_proba'):
                prob = np.max(self.ml_model.predict_proba(features_scaled))
            else:
                prob = conf
            
            # Store result
            detections.append({
                'bbox': (x1, y1, x2, y2),
                'class': prediction,
                'confidence': prob,
                'yolo_conf': conf
            })
            
            # Draw on image
            color = (0, 255, 0) if prediction == 0 else (0, 0, 255)
            cv2.rectangle(result_img, (x1, y1), (x2, y2), color, 2)
            label = f"{prediction} ({prob:.2f})"
            cv2.putText(result_img, label, (x1, y1-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
        
        return result_img, detections

pipeline = HybridAIPipeline(yolo_model, ml_model, scaler)
print("✓ Hybrid AI Pipeline initialized")

## 5. Test Pipeline on Sample Images

In [ ]:
print("\n" + "="*60)
print("🧪 TESTING PIPELINE ON SAMPLE IMAGES")
print("="*60)

test_images_dir = BASE_DIR / 'data/raw/images'
test_images = sorted(list(test_images_dir.glob('*.jpg')) + list(test_images_dir.glob('*.png')))[:3]

print(f"\nTesting on {len(test_images)} images...\n")

for img_path in test_images:
    print(f"Processing: {img_path.name}")
    
    result_img, detections = pipeline.predict(img_path)
    
    if result_img is not None:
        print(f"  Detections: {len(detections)}")
        for i, det in enumerate(detections[:3]):
            print(f"    {i+1}. Class: {det['class']}, Confidence: {det['confidence']:.3f}")
    print()

## 6. Visualize Results

In [ ]:
# Hiển thị kết quả
if len(test_images) > 0:
    fig, axes = plt.subplots(1, min(3, len(test_images)), figsize=(15, 5))
    if len(test_images) == 1:
        axes = [axes]
    
    for idx, img_path in enumerate(test_images[:3]):
        result_img, detections = pipeline.predict(img_path)
        
        if result_img is not None:
            result_img_rgb = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
            axes[idx].imshow(result_img_rgb)
            axes[idx].set_title(f"{img_path.name}\n({len(detections)} detections)")
            axes[idx].axis('off')
    
    plt.suptitle('Hybrid AI Pipeline Results', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 7. Batch Processing & Save Results

## 8. Statistics & Summary

## 9. Save Pipeline Model for Deployment

## 10. Create Inference Script for Production

## 11. Final Summary